# Phase 4 catch-up – the approved-projects tracker in SQL

I load the corrected tracker (`data/tracker_corrected.csv`, re-checked in Phase 7A) into a SQLite database (`tracker.db`). Then I answer five questions with plain SQL.

SQLite comes with Python (`sqlite3`), so there is nothing extra to install. `pandas.read_sql` runs a query and returns a table.

In [1]:
import sqlite3, pandas as pd
tk = pd.read_csv('../data/tracker_corrected.csv')
con = sqlite3.connect('../data/tracker.db')
tk.to_sql('projects', con, if_exists='replace', index=False)
sql = lambda s: pd.read_sql(s, con)
print(sql("SELECT COUNT(*) AS units, SUM(investment_rs_cr) AS total_rs_cr FROM projects"))

   units  total_rs_cr
0     12       164293


### Q1. Investment by state
Where is the money going?

```sql
SELECT state, COUNT(*) AS units, SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(100.0 * SUM(investment_rs_cr) / (SELECT SUM(investment_rs_cr) FROM projects), 1) AS share_pct
FROM projects GROUP BY state ORDER BY investment_rs_cr DESC
```

In [2]:
sql("""SELECT state, COUNT(*) AS units, SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(100.0 * SUM(investment_rs_cr) / (SELECT SUM(investment_rs_cr) FROM projects), 1) AS share_pct
FROM projects GROUP BY state ORDER BY investment_rs_cr DESC""")

,state,units,investment_rs_cr,share_pct
0,Gujarat,6,128869,78.4
1,Assam,1,27120,16.5
2,Odisha,2,4011,2.4
3,Uttar Pradesh,1,3706,2.3
4,Andhra Pradesh,1,469,0.3
5,Punjab,1,118,0.1


### Q2. Investment by type of unit
Fab vs packaging vs compound.

```sql
SELECT unit_type, COUNT(*) AS units, SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(AVG(investment_rs_cr)) AS avg_rs_cr
FROM projects GROUP BY unit_type ORDER BY investment_rs_cr DESC
```

In [3]:
sql("""SELECT unit_type, COUNT(*) AS units, SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(AVG(investment_rs_cr)) AS avg_rs_cr
FROM projects GROUP BY unit_type ORDER BY investment_rs_cr DESC""")

,unit_type,units,investment_rs_cr,avg_rs_cr
0,Silicon fab,1,91526,91526.0
1,OSAT,6,43054,7176.0
2,ATMP (memory),1,22516,22516.0
3,Compound fab + ATMP,2,5135,2568.0
4,Advanced packaging,1,1944,1944.0
5,Discrete fab/ATMP,1,118,118.0


### Q3. Investment by status
How much is actually producing?

```sql
SELECT status, COUNT(*) AS units, SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(100.0 * SUM(investment_rs_cr) / (SELECT SUM(investment_rs_cr) FROM projects), 1) AS share_pct
FROM projects GROUP BY status ORDER BY investment_rs_cr DESC
```

In [4]:
sql("""SELECT status, COUNT(*) AS units, SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(100.0 * SUM(investment_rs_cr) / (SELECT SUM(investment_rs_cr) FROM projects), 1) AS share_pct
FROM projects GROUP BY status ORDER BY investment_rs_cr DESC""")

,status,units,investment_rs_cr,share_pct
0,Under construction,2,95232,58.0
1,Commercial production,3,33407,20.3
2,Pilot production,2,27988,17.0
3,Approved - start not confirmed,5,7666,4.7


### Q4. Top-2 concentration
How dependent is the programme on two companies?

```sql
SELECT CASE WHEN company LIKE 'Tata%' THEN 'Tata (fab + Assam)'
            WHEN company LIKE 'Micron%' THEN 'Micron' ELSE 'All others' END AS grp,
       SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(100.0 * SUM(investment_rs_cr) / (SELECT SUM(investment_rs_cr) FROM projects), 1) AS share_pct
FROM projects GROUP BY grp ORDER BY investment_rs_cr DESC
```

In [5]:
sql("""SELECT CASE WHEN company LIKE 'Tata%' THEN 'Tata (fab + Assam)'
            WHEN company LIKE 'Micron%' THEN 'Micron' ELSE 'All others' END AS grp,
       SUM(investment_rs_cr) AS investment_rs_cr,
       ROUND(100.0 * SUM(investment_rs_cr) / (SELECT SUM(investment_rs_cr) FROM projects), 1) AS share_pct
FROM projects GROUP BY grp ORDER BY investment_rs_cr DESC""")

,grp,investment_rs_cr,share_pct
0,Tata (fab + Assam),118646,72.2
1,All others,23131,14.1
2,Micron,22516,13.7


### Q5. Units in production by approval year
Does an earlier approval mean production now?

```sql
SELECT approval_year, COUNT(*) AS approved,
       SUM(CASE WHEN status = 'Commercial production' THEN 1 ELSE 0 END) AS in_commercial_production,
       SUM(CASE WHEN status = 'Pilot production' THEN 1 ELSE 0 END) AS in_pilot
FROM projects GROUP BY approval_year ORDER BY approval_year
```

In [6]:
sql("""SELECT approval_year, COUNT(*) AS approved,
       SUM(CASE WHEN status = 'Commercial production' THEN 1 ELSE 0 END) AS in_commercial_production,
       SUM(CASE WHEN status = 'Pilot production' THEN 1 ELSE 0 END) AS in_pilot
FROM projects GROUP BY approval_year ORDER BY approval_year""")

,approval_year,approved,in_commercial_production,in_pilot
0,2023,1,1,0
1,2024,4,2,1
2,2025,5,0,0
3,2026,2,0,1


**Reading this:**
- **Concentration:** Gujarat holds most of the investment, and Tata plus Micron hold about 86% of it.
- **Production:** only three units, mostly Micron by value, are in commercial production in 2026. Most investment is still under construction or not yet started.
- **Approval year vs production:** units approved in 2023–24 are the ones producing. Nothing approved in 2025–26 is in commercial production yet (Suchi's pilot line predates its approval).

This is the same picture as Phase 4, now reproducible with one query each.

In [7]:
con.close()